In [1]:
from pathlib import Path 

import numpy as np
import pandas as pd
import sqlalchemy as db
from tqdm.auto import tqdm
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models

/home/amos/anaconda3/envs/face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)  

In [23]:
collections = [x.name for x in CLIENT.get_collections().collections]
if 'FacialEmbeddings' not in collections:
        CLIENT.recreate_collection(collection_name='FacialEmbeddings',
                                   vectors_config=VectorParams(size=128, distance=Distance.COSINE))

/tmp/ipykernel_490510/3511449327.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  CLIENT.recreate_collection(collection_name='FacialEmbeddings',


In [4]:
username = 'amos'
password = 'M0$hicat'
host = '192.168.0.131'
port = '3306'
database = 'CineFace'

In [5]:
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [10]:
df = pd.read_csv('/home/amos/datasets/shining_bat_encodings.csv', index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,face_num,img_width,img_height,filename,filepath,distance_from_center,pct_of_frame,encoding,series_id,episode_id
0,479,94,695,397,535,208,634,196,589,237,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.63,0.0708,-0.0674933\n0.0943281\n0.0449327\n0.00756572\n...,81505,15659200
1,515,114,730,422,570,231,669,223,623,276,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,27.04,0.0715,-0.0405336\n0.0716574\n0.0567907\n0.00282796\n...,81505,15659200
2,509,108,721,410,562,226,662,216,616,264,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.74,0.0695,-0.0331698\n0.0957054\n0.0514582\n0.0187592\n-...,81505,15659200
3,515,99,732,407,584,220,685,212,648,261,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,27.02,0.0722,-0.0560095\n0.0887332\n0.0417233\n0.0227062\n-...,81505,15659200
4,492,117,709,416,550,233,651,220,610,268,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.66,0.0698,-0.0416598\n0.103417\n0.0755524\n0.000812375\n...,81505,15659200


In [11]:
def parse_vector(vector):
    return np.array([float(x) for x in vector.split('\n')])

In [24]:
df['encoding'] = df['encoding'].map(parse_vector)
CLIENT.upload_points(collection_name='FacialEmbeddings',
              points=[
              PointStruct(
                id=idx,
                vector=row['encoding'].tolist(),
                payload={'series_id': row['series_id'],
                         'episode_id': row['episode_id'],
                         'frame_num': row['frame_num'],
                         'face_num': row['face_num']}
            )
            for idx, row in df.iterrows()]
)

In [21]:
cnt = CLIENT.count(collection_name='FacialEmbeddings').count
vectors = CLIENT.retrieve('FacialEmbeddings', [x for x in range(cnt)], with_vectors=True, with_payload=True)
vectors[0]

Record(id=268, payload={}, vector=[-0.010243356, 0.05097137, 0.061081912, -0.0076216003, -0.091094606, -0.04637473, -0.0076414184, 0.0016029114, 0.10875134, -0.017717883, 0.15278438, -0.004398243, -0.19139788, -0.015808856, 0.020868702, 0.06611022, -0.10394859, -0.050856013, -0.08812298, -0.105368264, -0.026518026, 0.045311972, -0.003634271, -0.02196722, -0.045601904, -0.19852087, -0.061171893, -0.10894284, 0.12072785, -0.0851898, 0.0066690403, 0.005932662, -0.09750237, -0.04789546, 0.01647724, -0.018545616, -0.010405857, -0.07784378, 0.14147311, -0.034457833, -0.11066784, -0.027330996, 0.014418788, 0.19197775, 0.14259669, 0.0021426573, 0.008637414, -0.047302596, 0.091370694, -0.18871695, 0.034271795, 0.11013872, 0.063333936, 0.09050782, 0.060963325, -0.14892372, 0.027413055, 0.10124844, -0.13836998, 0.04554484, 0.06010206, -0.012130618, -0.03168808, -0.080473185, 0.16803475, 0.02527308, -0.075610675, -0.108428344, 0.08655564, -0.14907522, 0.0009482683, 0.08107458, -0.09090619, -0.1046

In [26]:
dict(vectors[0].payload)

{}

In [129]:
(10000000 * 1.5 * 128 * 4)/1024/1024/1024

7.152557373046875